# Timeseries data

In this example we calculate the data of a wind farm with 67 turbines in a time series containing 8000 uniform inflow states.

The required imports are:

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

import foxes
import foxes.variables as FV

First, we create the `states`. The `data_source` can be any csv-type file (or `pandas` readable equivalent), or a `pandas.DataFrame` object. If it is a file path, then it will first be searched in the file system, and if not found, in the static data. If it is also not found there, an error showing the available static data file names is displayed.

In this example the static data file `timeseries_8000.csv.gz` will be used, with content
```
Time,ws,wd,ti
2017-01-01 00:00:00,15.62,244.06,0.0504
2017-01-01 00:30:00,15.99,243.03,0.0514
2017-01-01 01:00:00,16.31,243.01,0.0522
2017-01-01 01:30:00,16.33,241.26,0.0523
...
```
Notice the column names, and how they appear in the `Timeseries` constructor:

In [ ]:
states = foxes.input.states.Timeseries(
    data_source="timeseries_8000.csv.gz",
    output_vars=[FV.WS, FV.WD, FV.TI, FV.RHO],
    var2col={FV.WS: "ws", FV.WD: "wd", FV.TI: "ti"},
    fixed_vars={FV.RHO: 1.225},
)

We can visualize the wind distribution via the `StatesRosePlotOutput`. Here we display the ambient wind speed in a wind rose with 16 wind direction sectors and 5 wind speed bins: 

In [ ]:
rose_farm = foxes.WindFarm()
rose_farm.add_turbine(
    foxes.Turbine([0.0, 0.0], H=100.0, D=1.0, turbine_models=["null_type"]), verbosity=0
)
foxes.algorithms.Downwind(rose_farm, states, wake_models=[], verbosity=0).calc_outputs(
    outputs=foxes.models.outputs.StatesRosePlotOutput(
        point=[0.0, 0.0, 100.0], plot=True, plot_pars=dict(figsize=(6, 6))
    )
)[0]
plt.show()

For the time series with uniform data, any choice of the `point` argument will produce the same figure.

Next, we create the example wind farm with 67 turbines from static data. The file `test_farm_67.csv` has the following structure:
```
index,label,x,y
0,T0,101872.70,1004753.57
1,T1,103659.97,1002993.29
2,T2,100780.09,1000779.97
3,T3,100290.42,1004330.88
...
```
For more options, check the API section `foxes.input.farm_layout`. 

We consider two turbine models in this example: the wind turbine type `NREL5MW` and the turbine model `kTI_02`, both from the default model book. The latter model adds the variable `k` for each state and turbine, calculated as `k = kTI * TI`, with constant `kTI = 0.2`. The parameter `k` will later be used by the wake model.

In [ ]:
farm = foxes.WindFarm()
foxes.input.farm_layout.add_from_file(
    farm, "test_farm_67.csv", turbine_models=["NREL5MW"], verbosity=0
)

Next, create the `algorithm` with `Bastankhah2014_quadratic_ka02` for wind-speed deficits and `CrespoHernandez_max_ka04` for turbulence intensity:

In [ ]:
algo = foxes.algorithms.Downwind(
    farm,
    states,
    rotor_model="centre",
    wake_models=["Bastankhah2014_quadratic_ka02", "CrespoHernandez_max_ka04"],
    verbosity=0,
)

Also notice the `chunks` parameter, specifying that always 1000 states should be considered in vectorized form during calculations. The progress is automatically visualized when invoking the `DaskRunner`:

In [ ]:
algo.mbook.outputs["farm_results"].fdata_variables.extend(
    [
        FV.CAP,
        FV.CAPF,
        FV.AMB_CAPF,
        FV.EFF,
        FV.YLD,
        FV.AMB_YLD,
    ]
)

farm_results = algo.calc_farm()

fr = farm_results.to_dataframe()
print("\n", fr[[FV.WD, FV.AMB_REWS, FV.REWS, FV.AMB_P, FV.P]])

The selected turbine metrics are evaluated automatically before output:

In [ ]:
# Metrics are evaluated automatically from the output selection.
print(
    "\nState-turbine metrics:\n",
    farm_results[[FV.CAP, FV.CAPF, FV.AMB_CAPF, FV.EFF, FV.YLD, FV.AMB_YLD]],
)

We can visualize the mean rotor equivalent wind speed as seen by each turbine and also the mean efficiency with respect to the time series data as colored layout plots:

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(14, 5))

foxes.algorithms.Downwind(
    farm,
    foxes.input.states.SingleStateStates(ws=8, wd=270, ti=0.08, rho=1.225),
    wake_models=[],
    verbosity=0,
).calc_outputs(
    outputs=foxes.models.outputs.FarmLayoutOutput(
        operation="get_figure",
        parameters=dict(
            fig=fig,
            ax=axs[0],
            color_by="mean_REWS",
            title="Mean REWS [m/s]",
            s=150,
            annotate=0,
        ),
    ),
    farm_results=farm_results,
)
foxes.algorithms.Downwind(
    farm,
    foxes.input.states.SingleStateStates(ws=8, wd=270, ti=0.08, rho=1.225),
    wake_models=[],
    verbosity=0,
).calc_outputs(
    outputs=foxes.models.outputs.FarmLayoutOutput(
        operation="get_figure",
        parameters=dict(
            fig=fig,
            ax=axs[1],
            color_by="mean_EFF",
            title="Mean efficiency [%]",
            s=150,
            annotate=0,
        ),
    ),
    farm_results=farm_results,
)
plt.show()